In [1]:
# -*- coding: utf-8 -*-
"""
1D adapted SA-PINN benchmark with strict LHS-based evaluation timing and error computation.

This version is aligned with the DAE/DAE-RAR timing protocol:
1. T_eval is measured on the same fixed LHS testing set used for e2/einf.
2. The LHS testing set guarantees exactly NUM_SAMPLES unique reference-grid points.
3. Error computation is performed after T_eval timing and is NOT included in T_eval.
4. T_eval is averaged over repeated reconstructions after GPU warm-up.
5. Loss history is stored during training without per-epoch CPU transfer and is
   moved to CPU only after T_train timing.
6. SA-PINN is trained separately for each mu because the PDE residual and IC depend on mu.
"""

import time
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.autograd import grad
from scipy.stats import qmc
from scipy.spatial import cKDTree


# =============================================================================
# Basic settings
# =============================================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = False

SEEDS = [33, 99, 202, 1234, 5678, 9999]
MU_LIST = [0.0001]

X_MIN, X_MAX, T_FINAL = 0.0, 1.0, 0.3
L_VALUE, R_VALUE, H0_VALUE = -10.0, 5.0, 0.1

DEPTH, WIDTH = 4, 10
EPOCHS = 20000
N_F, N_I = 2000, 2000
TOTAL_POINTS = N_F + N_I

METHOD_NAME = "SAPINN"

# LHS testing settings
NUM_SAMPLES = 5000
LHS_SEED = 1234

# Repeated timing settings for T_eval
EVAL_WARMUP = 20
EVAL_REPEAT = 200

# Optional full-grid output for visualization only
SAVE_FULL_GRID = False
NX_FULL, NT_FULL = 201, 201

BASE_PATH = "."


# =============================================================================
# Utilities
# =============================================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def source_f(x):
    return x - x**2 + x**3


def u_init(x, mu):
    return 7.5 * torch.tanh((x - H0_VALUE) / mu) - 2.5


def mean_std(values):
    arr = np.asarray(values, dtype=float)
    if len(arr) <= 1:
        return np.nanmean(arr), 0.0
    return np.nanmean(arr), np.nanstd(arr, ddof=1)


# =============================================================================
# Network and SA-PINN model
# =============================================================================

class MLP(nn.Module):
    def __init__(self, in_dim, out_dim, width, depth):
        super().__init__()
        layers = [nn.Linear(in_dim, width), nn.Tanh()]
        for _ in range(depth - 2):
            layers += [nn.Linear(width, width), nn.Tanh()]
        layers.append(nn.Linear(width, out_dim))
        self.net = nn.Sequential(*layers)
        self.reset_parameters()

    def reset_parameters(self):
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)


class AdaptedSAPINN(nn.Module):
    def __init__(self):
        super().__init__()
        self.N_phi = MLP(1, 2, WIDTH, DEPTH)
        self.N_h = MLP(1, 1, WIDTH, DEPTH)

    def h(self, t):
        return H0_VALUE + t * self.N_h(t)

    def outer(self, x):
        raw = self.N_phi(x)
        phi_m = L_VALUE + (x - X_MIN) * raw[:, 0:1]
        phi_p = R_VALUE + (x - X_MAX) * raw[:, 1:2]
        return phi_m, phi_p

    def V_composite(self, x, t, mu):
        h_val = self.h(t)

        phi_m_x, phi_p_x = self.outer(x)
        phi_m_h, phi_p_h = self.outer(h_val)

        delta = phi_p_h - phi_m_h

        arg_l = (h_val - x) * delta / (2.0 * mu)
        arg_r = (x - h_val) * delta / (2.0 * mu)

        Q_l = delta * torch.sigmoid(-arg_l)
        Q_r = -delta * torch.sigmoid(-arg_r)

        return torch.where(x <= h_val, phi_m_x + Q_l, phi_p_x + Q_r)

    def forward(self, x, t, mu):
        base = self.V_composite(x, t, mu)

        x_l = torch.full_like(x, X_MIN)
        x_r = torch.full_like(x, X_MAX)

        u_l = self.V_composite(x_l, t, mu)
        u_r = self.V_composite(x_r, t, mu)

        ell_l = (X_MAX - x) / (X_MAX - X_MIN)
        ell_r = (x - X_MIN) / (X_MAX - X_MIN)

        return base + ell_l * (L_VALUE - u_l) + ell_r * (R_VALUE - u_r)


# =============================================================================
# Loss and evaluation
# =============================================================================

def pde_residual(model, X_f, mu, create_graph=True):
    x = X_f[:, 0:1]
    t = X_f[:, 1:2]

    u = model(x, t, mu)
    grads = grad(
        u.sum(),
        X_f,
        create_graph=create_graph,
        retain_graph=create_graph
    )[0]
    u_x = grads[:, 0:1]
    u_t = grads[:, 1:2]
    u_xx = grad(
        u_x.sum(),
        X_f,
        create_graph=create_graph,
        retain_graph=create_graph
    )[0][:, 0:1]

    return mu * u_xx - u_t + u * u_x - source_f(x)


def compute_loss(model, X_f, X_i, U_i, mu):
    res = pde_residual(model, X_f, mu, create_graph=True)
    loss_pde = torch.mean(res**2)

    u_i = model(X_i[:, 0:1], X_i[:, 1:2], mu)
    loss_ic = torch.mean((u_i - U_i)**2)

    return loss_pde + loss_ic


def eval_u(model, x_eval, t_eval, mu):
    """Online SA-PINN reconstruction. This is exactly the part timed by T_eval."""
    with torch.no_grad():
        return model(x_eval, t_eval, mu)


# =============================================================================
# LHS testing set and error computation
# =============================================================================

def true_u0_filename(mu):
    return f"1d_U0_true_mu{mu:.0e}_201_201_Mathematica.csv"


def lhs_index_filename(mu):
    mu_label = f"{mu:.0e}"
    return f"1d_LHS_sample_indices_mu{mu_label}.npy"


def lhs_points_filename(mu):
    mu_label = f"{mu:.0e}"
    return f"1d_LHS_test_points_mu{mu_label}.csv"


def load_true_solution_u0(mu):
    filename = true_u0_filename(mu)
    path = os.path.join(BASE_PATH, filename)

    if not os.path.exists(path):
        raise FileNotFoundError(f"Cannot find true solution file: {filename}")

    df = pd.read_csv(path)
    df = df.sort_values(by=["t", "x"]).reset_index(drop=True)
    return df, filename


def generate_lhs_indices(df_true, mu):
    """Generate exactly NUM_SAMPLES unique LHS-selected reference-grid indices."""
    total_points = len(df_true)
    if total_points < NUM_SAMPLES:
        raise ValueError(
            f"Reference grid has only {total_points} points, "
            f"which is smaller than NUM_SAMPLES={NUM_SAMPLES}."
        )

    t_min, t_max = df_true["t"].min(), df_true["t"].max()
    x_min, x_max = df_true["x"].min(), df_true["x"].max()
    all_points = df_true[["t", "x"]].values
    kdtree = cKDTree(all_points)

    selected = []
    used = set()
    batch_id = 0
    max_batches = 100

    while len(selected) < NUM_SAMPLES and batch_id < max_batches:
        sampler = qmc.LatinHypercube(d=2, seed=LHS_SEED + batch_id)
        lhs_sample = sampler.random(n=NUM_SAMPLES)
        lhs_sample_scaled = qmc.scale(lhs_sample, [t_min, x_min], [t_max, x_max])
        _, candidate_indices = kdtree.query(lhs_sample_scaled)

        for idx in candidate_indices:
            idx = int(idx)
            if idx not in used:
                used.add(idx)
                selected.append(idx)
                if len(selected) == NUM_SAMPLES:
                    break
        batch_id += 1

    if len(selected) < NUM_SAMPLES:
        remaining = np.setdiff1d(
            np.arange(total_points),
            np.asarray(selected, dtype=int),
            assume_unique=False
        )
        if len(remaining) < NUM_SAMPLES - len(selected):
            raise RuntimeError(
                "Not enough remaining reference-grid points to complete "
                f"{NUM_SAMPLES} unique test points."
            )
        rng = np.random.default_rng(LHS_SEED)
        fill = rng.choice(remaining, size=NUM_SAMPLES - len(selected), replace=False)
        selected.extend([int(i) for i in fill])

    sample_indices = np.asarray(selected, dtype=int)

    if len(sample_indices) != NUM_SAMPLES:
        raise RuntimeError(
            f"Failed to generate exactly {NUM_SAMPLES} test points. "
            f"Got {len(sample_indices)}."
        )
    if len(np.unique(sample_indices)) != NUM_SAMPLES:
        raise RuntimeError("Generated LHS test indices are not unique.")

    np.save(lhs_index_filename(mu), sample_indices)
    return sample_indices


def build_or_load_lhs_test_set_from_true(mu):
    """
    Use exactly the same LHS indices as DAE/DAE-RAR if they already exist and are valid.
    Otherwise, generate exactly NUM_SAMPLES unique test indices and save them.
    """
    df_true, filename = load_true_solution_u0(mu)
    index_file = lhs_index_filename(mu)

    if os.path.exists(index_file):
        sample_indices = np.load(index_file)
        valid_existing = (
            len(sample_indices) == NUM_SAMPLES
            and len(np.unique(sample_indices)) == NUM_SAMPLES
            and np.min(sample_indices) >= 0
            and np.max(sample_indices) < len(df_true)
        )
        if valid_existing:
            print(f"[mu={mu}] Loaded valid LHS indices from {index_file}.")
        else:
            print(
                f"[mu={mu}] Existing LHS index file is invalid "
                f"(length={len(sample_indices)}, unique={len(np.unique(sample_indices))}). "
                f"Regenerating..."
            )
            sample_indices = generate_lhs_indices(df_true, mu)
    else:
        sample_indices = generate_lhs_indices(df_true, mu)
        print(f"[mu={mu}] Generated and saved exactly {len(sample_indices)} LHS indices to {index_file}.")

    if len(sample_indices) != NUM_SAMPLES or len(np.unique(sample_indices)) != NUM_SAMPLES:
        raise RuntimeError(f"LHS test set for mu={mu} is not exactly {NUM_SAMPLES} unique points.")

    t_lhs_np = df_true.iloc[sample_indices]["t"].values.reshape(-1, 1)
    x_lhs_np = df_true.iloc[sample_indices]["x"].values.reshape(-1, 1)
    true_lhs_np = df_true.iloc[sample_indices].iloc[:, 2].values.reshape(-1)

    t_lhs = torch.tensor(t_lhs_np, dtype=torch.float32, device=DEVICE)
    x_lhs = torch.tensor(x_lhs_np, dtype=torch.float32, device=DEVICE)

    df_test = pd.DataFrame({
        "t": t_lhs_np.reshape(-1),
        "x": x_lhs_np.reshape(-1),
        "u_true": true_lhs_np
    })
    df_test.to_csv(lhs_points_filename(mu), index=False)

    print(f"[mu={mu}] LHS test set built from {filename}: N_test={len(sample_indices)}")

    return {
        "df_true": df_true,
        "sample_indices": sample_indices,
        "t_lhs": t_lhs,
        "x_lhs": x_lhs,
        "t_lhs_np": t_lhs_np,
        "x_lhs_np": x_lhs_np,
        "true_lhs_np": true_lhs_np,
        "n_test": len(sample_indices)
    }


def compute_error(true_u, pred_u):
    diff = pred_u - true_u
    e2 = np.linalg.norm(diff) / np.linalg.norm(true_u)
    einf = np.max(np.abs(diff))
    return e2, einf


# =============================================================================
# Optional full-grid output
# =============================================================================

def save_full_grid_prediction(model, mu, seed):
    x_vals = np.linspace(X_MIN, X_MAX, NX_FULL)
    t_vals = np.linspace(0.0, T_FINAL, NT_FULL)
    T_grid, X_grid = np.meshgrid(t_vals, x_vals, indexing="ij")

    x_eval = torch.tensor(X_grid.reshape(-1, 1), dtype=torch.float32, device=DEVICE)
    t_eval = torch.tensor(T_grid.reshape(-1, 1), dtype=torch.float32, device=DEVICE)

    u_tensor = eval_u(model, x_eval, t_eval, mu)
    u_pred = u_tensor.detach().cpu().numpy().reshape(-1)

    df = pd.DataFrame({
        "x": X_grid.reshape(-1),
        "t": T_grid.reshape(-1),
        "u": u_pred
    })
    df.to_csv(f"1d_{METHOD_NAME}_U0_predicted_mu{mu:.0e}_seed{seed}.csv", index=False)


# =============================================================================
# Main program
# =============================================================================

print("\n" + "=" * 80)
print(f"Starting 1D {METHOD_NAME} benchmark with strict LHS-based T_eval and error")
print("=" * 80 + "\n")

print(f"Device: {DEVICE}")
print(f"LHS test points: N_test={NUM_SAMPLES}, LHS seed={LHS_SEED}")
print(f"T_eval warmup: {EVAL_WARMUP}, repeated timing: {EVAL_REPEAT}")
print(f"TOTAL_POINTS per iter: {TOTAL_POINTS}\n")

lhs_data = {}
for mu in MU_LIST:
    lhs_data[mu] = build_or_load_lhs_test_set_from_true(mu)

metrics = {mu: [] for mu in MU_LIST}

for mu in MU_LIST:
    print("\n" + "=" * 80)
    print(f"Starting 1D {METHOD_NAME} for mu={mu}")
    print("=" * 80)

    data_mu = lhs_data[mu]
    x_eval_lhs = data_mu["x_lhs"]
    t_eval_lhs = data_mu["t_lhs"]
    true_lhs_np = data_mu["true_lhs_np"]
    n_test = data_mu["n_test"]

    if n_test != NUM_SAMPLES:
        raise RuntimeError(f"N_test mismatch for mu={mu}: expected {NUM_SAMPLES}, got {n_test}.")

    for seed in SEEDS:
        print("\n" + "-" * 80)
        print(f"Running mu={mu}, seed={seed}")
        print("-" * 80)

        set_seed(seed)

        X_f = torch.cat([
            torch.empty(N_F, 1, device=DEVICE).uniform_(X_MIN, X_MAX),
            torch.empty(N_F, 1, device=DEVICE).uniform_(0.0, T_FINAL),
        ], dim=1).requires_grad_(True)

        x_i = torch.empty(N_I, 1, device=DEVICE).uniform_(X_MIN, X_MAX)
        X_i = torch.cat([x_i, torch.zeros_like(x_i)], dim=1)
        U_i = u_init(x_i, mu).detach()

        model = AdaptedSAPINN().to(DEVICE)
        optimizer = optim.Adam(model.parameters(), lr=1e-3)

        # Store detached GPU scalar tensors during training.
        # This avoids per-epoch CPU-GPU synchronization caused by .cpu().item().
        loss_list = []

        model.train()
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        train_start = time.perf_counter()

        for epoch in range(EPOCHS):
            optimizer.zero_grad(set_to_none=True)

            loss = compute_loss(model, X_f, X_i, U_i, mu)

            loss.backward()
            optimizer.step()

            loss_list.append(loss.detach())

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        T_train = time.perf_counter() - train_start

        loss_history = torch.stack(loss_list).detach().cpu().numpy().astype(np.float64)
        e_loss = float(loss_history[-1])

        T_train_per_iter_ms = T_train * 1e3 / EPOCHS
        T_train_per_iter_point_us = T_train * 1e6 / (EPOCHS * TOTAL_POINTS)

        print(
            f" > trained: epochs={EPOCHS}, points={TOTAL_POINTS}, "
            f"T_train={T_train:.2f}s, e_loss={e_loss:.3e}"
        )

        np.save(
            f"1d_{METHOD_NAME}_loss_history_mu{mu:.0e}_seed{seed}.npy",
            loss_history
        )

        model.eval()

        # Warm-up on the same LHS testing set. This is outside T_eval.
        with torch.no_grad():
            for _ in range(EVAL_WARMUP):
                _ = eval_u(model, x_eval_lhs, t_eval_lhs, mu)

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        # Repeated timing on the same LHS testing set.
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        eval_start = time.perf_counter()

        with torch.no_grad():
            for _ in range(EVAL_REPEAT):
                _ = eval_u(model, x_eval_lhs, t_eval_lhs, mu)

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        T_eval = (time.perf_counter() - eval_start) / EVAL_REPEAT

        # Compute prediction once more for error. This is outside T_eval.
        with torch.no_grad():
            u_eval_tensor = eval_u(model, x_eval_lhs, t_eval_lhs, mu)

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        u_pred_lhs = u_eval_tensor.detach().cpu().numpy().reshape(-1)
        e2, einf = compute_error(true_lhs_np, u_pred_lhs)

        T_total = T_train + T_eval

        metrics[mu].append({
            "Seed": seed,
            "N_test": n_test,
            "e_loss": e_loss,
            "e2": e2,
            "einf": einf,
            "T_train": T_train,
            "T_eval": T_eval,
            "T_total": T_total,
            "T_train_per_iter_ms": T_train_per_iter_ms,
            "T_train_per_iter_point_us": T_train_per_iter_point_us,
            "total_trained_steps": EPOCHS,
            "total_point_steps": EPOCHS * TOTAL_POINTS,
            "final_residual_points": TOTAL_POINTS,
            "eval_warmup": EVAL_WARMUP,
            "eval_repeat": EVAL_REPEAT
        })

        print(
            f"    -> [mu={mu}] N_test={n_test}, T_eval={T_eval:.6e}s, "
            f"e2={e2:.3e}, einf={einf:.3e}"
        )

        mu_label = f"{mu:.0e}"
        df_lhs_pred = pd.DataFrame({
            "t": data_mu["t_lhs_np"].reshape(-1),
            "x": data_mu["x_lhs_np"].reshape(-1),
            "u": u_pred_lhs
        })
        df_lhs_pred.to_csv(
            f"1d_{METHOD_NAME}_U0_predicted_LHS_mu{mu_label}_seed{seed}.csv",
            index=False
        )

        if SAVE_FULL_GRID:
            save_full_grid_prediction(model, mu, seed)


# =============================================================================
# Summary tables
# =============================================================================

print("\n" + "=" * 80)
print("ALL SEEDS COMPLETED. GENERATING SUMMARY TABLES.")
print("=" * 80 + "\n")

for mu in MU_LIST:
    dfm = pd.DataFrame(metrics[mu])
    mu_label = f"{mu:.0e}"
    dfm.to_csv(f"1d_{METHOD_NAME}_mu{mu_label}_Metrics_Summary.csv", index=False)

    cols = [
        "e_loss",
        "e2",
        "einf",
        "T_train",
        "T_eval",
        "T_total",
        "T_train_per_iter_ms",
        "T_train_per_iter_point_us",
        "total_trained_steps",
        "total_point_steps",
        "final_residual_points",
        "N_test"
    ]

    stats = {c: mean_std(dfm[c].values) for c in cols}

    print(f"### Results for 1D {METHOD_NAME}, mu={mu} [Mean \\pm Sample Std] ###")
    print(f"N_test: {stats['N_test'][0]:.0f} \\pm {stats['N_test'][1]:.0f}")
    print(f"e_loss: {stats['e_loss'][0]:.3e} \\pm {stats['e_loss'][1]:.3e}")
    print(f"e_2: {stats['e2'][0]:.3e} \\pm {stats['e2'][1]:.3e}")
    print(f"e_inf: {stats['einf'][0]:.3e} \\pm {stats['einf'][1]:.3e}")
    print(f"T_train (s): {stats['T_train'][0]:.2f} \\pm {stats['T_train'][1]:.2f}")
    print(f"T_eval (s): {stats['T_eval'][0]:.6e} \\pm {stats['T_eval'][1]:.6e}")
    print(f"T_total (s): {stats['T_total'][0]:.2f} \\pm {stats['T_total'][1]:.2f}")
    print(
        f"T_train/iter (ms): "
        f"{stats['T_train_per_iter_ms'][0]:.4f} \\pm "
        f"{stats['T_train_per_iter_ms'][1]:.4f}"
    )
    print(
        f"T_train/(iter*pt) (us): "
        f"{stats['T_train_per_iter_point_us'][0]:.4f} \\pm "
        f"{stats['T_train_per_iter_point_us'][1]:.4f}"
    )
    print(
        f"Optimization steps: "
        f"{stats['total_trained_steps'][0]:.1f} \\pm "
        f"{stats['total_trained_steps'][1]:.1f}"
    )
    print(
        f"Point-iterations: "
        f"{stats['total_point_steps'][0]:.1f} \\pm "
        f"{stats['total_point_steps'][1]:.1f}"
    )
    print(
        f"Final residual points: "
        f"{stats['final_residual_points'][0]:.1f} \\pm "
        f"{stats['final_residual_points'][1]:.1f}"
    )
    print()



Starting 1D SAPINN benchmark with strict LHS-based T_eval and error

Device: cuda
LHS test points: N_test=5000, LHS seed=1234
T_eval warmup: 20, repeated timing: 200
TOTAL_POINTS per iter: 4000

[mu=0.0001] Loaded valid LHS indices from 1d_LHS_sample_indices_mu1e-04.npy.
[mu=0.0001] LHS test set built from 1d_U0_true_mu1e-04_201_201_Mathematica.csv: N_test=5000

Starting 1D SAPINN for mu=0.0001

--------------------------------------------------------------------------------
Running mu=0.0001, seed=33
--------------------------------------------------------------------------------
 > trained: epochs=20000, points=4000, T_train=1403.90s, e_loss=2.279e+03
    -> [mu=0.0001] N_test=5000, T_eval=2.804845e-03s, e2=7.096e-01, einf=1.359e+01

--------------------------------------------------------------------------------
Running mu=0.0001, seed=99
--------------------------------------------------------------------------------
 > trained: epochs=20000, points=4000, T_train=1408.95s, e_loss=

In [2]:
pip install scipy -i https://pypi.tuna.tsinghua.edu.cn/simple

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.7/37.7 MB 11.6 MB/s eta 0:00:0000:010:01m
Note: you may need to restart the kernel to use updated packages.
